# Webアクセスログ分析デモ（PySpark）

Cloudera AI Workbench の **JupyterLab** 上で、HDFS に蓄積された Apache httpd ログ（`/data/logs/httpd`）を **そのまま** 読み込み、フィルタと集計を行います。

**デモのポイント**
- ターミナルの `spark-submit` ではなく、**セルを順に実行**するだけで分析できる
- 結果は **表** と **グラフ** を Notebook 内にその場表示
- 事前の ETL やテーブル取り込みは不要（デモ2と同じ考え方）

**想定**: CDP 上で Kerberos 済みのセッション、Spark 対応 Python カーネル（環境により `spark` が既に用意されている場合があります）

## 1. Spark セッションとデータの場所

In [6]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# Workbench では spark が既にあることが多いです。なければ作成します。
try:
    spark
except NameError:
    spark = SparkSession.builder.appName("HttpdLogNotebook").getOrCreate()

spark.sparkContext.setLogLevel("WARN")

# NiFi が書き込む HDFS パス（デモ2と同じ）
LOG_PATH = "/data/logs/httpd/access.log_*"
print("Reading:", LOG_PATH)

Reading: /data/logs/httpd/access.log_*


## 2. ログを読み込んで列に分ける（スキーマオンリード）

1 行 = 1 リクエスト。正規表現で `client_ip` や `status` などに分解します。

In [12]:
COMBINED_LOG = (
    r"^(\S+)\s+\S+\s+(\S+)\s+\[([^\]]+)\]\s+"
    r'"(\S+)\s+(\S+)\s+(\S+)"\s+(\d+)\s+(\d+)'
)

raw = spark.read.text(LOG_PATH)
logs = (
    raw.withColumn("line", F.col("value"))
    .withColumn("client_ip", F.regexp_extract("value", COMBINED_LOG, 1))
    .withColumn("authuser", F.regexp_extract("value", COMBINED_LOG, 2))
    .withColumn("log_date", F.regexp_extract("value", COMBINED_LOG, 3))
    .withColumn("method", F.regexp_extract("value", COMBINED_LOG, 4))
    .withColumn("path", F.regexp_extract("value", COMBINED_LOG, 5))
    .withColumn("status", F.regexp_extract("value", COMBINED_LOG, 7).cast("int"))
    .filter(F.col("client_ip") != "")
)

print("Log lines:", logs.count())

Log lines: 2160


## 3. 先頭 10 件を表で確認

Notebook では `.toPandas()` で **見やすい表** として表示できます（件数が少ない Preview 向け）。

In [13]:
preview = logs.select(
    "client_ip", "authuser", "method", "path", "status", "log_date"
).limit(10)

preview.toPandas()

,client_ip,authuser,method,path,status,log_date
0,234.2.126.32,-,POST,/search,200,04/Oct/2026:18:35:20 +0000
1,7.37.120.49,-,PATCH,/api/v1/users,401,03/Oct/2026:16:49:20 +0100
2,162.190.49.227,-,OPTIONS,/,503,04/Oct/2026:04:29:26 -0700
3,238.250.43.242,-,PUT,/health,302,03/Oct/2026:14:14:49 -0700
4,188.165.172.174,admin,GET,/static/css/main.css,500,04/Oct/2026:13:11:53 -0500
5,101.183.102.222,-,DELETE,/images/logo.png,502,03/Oct/2026:14:00:21 -0700
6,117.195.238.232,-,OPTIONS,/login,401,04/Oct/2026:19:00:11 -0500
7,190.157.214.6,admin,POST,/index.html,401,03/Oct/2026:11:54:24 -0500
8,48.176.183.196,user51,GET,/,204,03/Oct/2026:03:32:56 -0500
9,124.27.58.109,-,HEAD,/index.html,404,04/Oct/2026:20:51:59 +0000


## 4. フィルタ — HTTP 4xx / 5xx

条件を 1 行足すだけで調査対象を絞れます。

In [14]:
errors = (
    logs.filter(F.col("status") >= 400)
    .select("client_ip", "method", "path", "status", "authuser", "log_date")
    .orderBy(F.col("status").desc())
)

errors.limit(20).toPandas()

,client_ip,method,path,status,authuser,log_date
0,214.59.61.143,PUT,/api/v1/orders,503,admin,04/Oct/2026:01:41:45 +0100
1,219.173.20.72,DELETE,/logout,503,admin,04/Oct/2026:18:14:21 -0700
2,8.247.145.50,POST,/api/v1/orders,503,-,04/Oct/2026:13:30:21 -0500
3,75.129.75.91,GET,/health,503,-,04/Oct/2026:04:39:01 +0100
4,55.18.12.236,HEAD,/api/v1/users,503,-,03/Oct/2026:05:56:54 +0000
5,209.204.230.164,PUT,/static/js/app.js,503,-,04/Oct/2026:06:03:32 +0000
6,136.226.28.56,OPTIONS,/static/js/app.js,503,-,02/Oct/2026:23:02:18 -0700
7,246.12.188.168,OPTIONS,/api/v1/users,503,admin,04/Oct/2026:01:39:23 +0900
8,41.151.65.254,PUT,/static/css/main.css,503,-,03/Oct/2026:00:57:23 +0100
9,5.229.35.109,PUT,/,503,-,03/Oct/2026:08:47:05 +0000


## 5. フィルタ — 注目パターン（admin / curl / 403）

In [15]:
suspicious = logs.filter(
    (F.col("authuser") == "admin")
    | F.col("line").contains("curl/")
    | (F.col("status") == 403)
)

suspicious.select(
    "client_ip", "authuser", "method", "path", "status", "log_date"
).limit(20).toPandas()

,client_ip,authuser,method,path,status,log_date
0,188.165.172.174,admin,GET,/static/css/main.css,500,04/Oct/2026:13:11:53 -0500
1,117.195.238.232,-,OPTIONS,/login,401,04/Oct/2026:19:00:11 -0500
2,190.157.214.6,admin,POST,/index.html,401,03/Oct/2026:11:54:24 -0500
3,105.239.46.235,-,PUT,/images/logo.png,403,04/Oct/2026:14:50:10 -0700
4,85.215.170.177,admin,POST,/,500,05/Oct/2026:00:09:53 -0700
5,220.196.137.163,user91,PUT,/api/v1/users,500,03/Oct/2026:17:02:10 +0100
6,100.81.184.63,admin,DELETE,/api/v1/orders,500,04/Oct/2026:22:34:35 +0100
7,97.252.227.248,-,PUT,/index.html,403,04/Oct/2026:17:34:04 -0500
8,29.108.100.243,-,POST,/,403,04/Oct/2026:14:16:30 +0900
9,143.7.104.140,-,POST,/login,403,04/Oct/2026:03:27:40 -0700


## 6. 集計 — ステータスコード別件数

In [11]:
by_status = (
    logs.groupBy("status")
    .agg(F.count("*").alias("requests"))
    .orderBy(F.col("requests").desc())
)

status_pdf = by_status.toPandas()
status_pdf

26/10/04 02:37:33 051 ERROR TaskSetManager: Task 0 in stage 7.0 failed 4 times; aborting job


Py4JJavaError: An error occurred while calling o157.collectToPython.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 0 in stage 7.0 failed 4 times, most recent failure: Lost task 0.3 in stage 7.0 (TID 97) (172.42.2.174 executor 53): java.io.FileNotFoundException: 
File does not exist: /data/logs/httpd/access.log_2026-10-03-20-35
	at org.apache.hadoop.hdfs.server.namenode.INodeFile.valueOf(INodeFile.java:87)
	at org.apache.hadoop.hdfs.server.namenode.INodeFile.valueOf(INodeFile.java:77)
	at org.apache.hadoop.hdfs.server.namenode.FSDirStatAndListingOp.getBlockLocations(FSDirStatAndListingOp.java:159)
	at org.apache.hadoop.hdfs.server.namenode.FSNamesystem.getBlockLocations(FSNamesystem.java:2040)
	at org.apache.hadoop.hdfs.server.namenode.NameNodeRpcServer.getBlockLocations(NameNodeRpcServer.java:738)
	at org.apache.hadoop.hdfs.protocolPB.ClientNamenodeProtocolServerSideTranslatorPB.getBlockLocations(ClientNamenodeProtocolServerSideTranslatorPB.java:454)
	at org

## 7. 集計 — アクセス元 IP Top 10

In [ ]:
top_ips = (
    logs.groupBy("client_ip")
    .agg(
        F.count("*").alias("requests"),
        F.sum(F.when(F.col("status") >= 400, 1).otherwise(0)).alias("errors"),
    )
    .orderBy(F.col("requests").desc())
    .limit(10)
)

ip_pdf = top_ips.toPandas()
ip_pdf

## 8. グラフで可視化

集計結果を **その場で図にする** 例です（matplotlib）。コードは短く、セル実行だけで画像が Notebook 内に残ります。

In [7]:
import matplotlib.pyplot as plt

status_pdf.plot.bar(x="status", y="requests", legend=False, color="steelblue", figsize=(7, 3))
plt.title("HTTP status code")
plt.tight_layout()
plt.show()

NameError: name 'status_pdf' is not defined

In [8]:
class_pdf = (
    logs.withColumn(
        "class",
        F.when(F.col("status") < 300, "2xx")
        .when(F.col("status") < 400, "3xx")
        .when(F.col("status") < 500, "4xx")
        .otherwise("5xx"),
    )
    .groupBy("class")
    .count()
    .toPandas()
)

class_pdf.plot.pie(y="count", labels=class_pdf["class"], autopct="%1.0f%%", legend=False, figsize=(5, 5))
plt.title("Status class (2xx–5xx)")
plt.ylabel("")
plt.show()

26/10/04 02:33:16 230 ERROR TaskSetManager: Task 0 in stage 5.0 failed 4 times; aborting job


Py4JJavaError: An error occurred while calling o121.collectToPython.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 0 in stage 5.0 failed 4 times, most recent failure: Lost task 0.3 in stage 5.0 (TID 85) (172.42.2.173 executor 52): java.io.FileNotFoundException: 
File does not exist: /data/logs/httpd/access.log_2026-10-03-20-30
	at org.apache.hadoop.hdfs.server.namenode.INodeFile.valueOf(INodeFile.java:87)
	at org.apache.hadoop.hdfs.server.namenode.INodeFile.valueOf(INodeFile.java:77)
	at org.apache.hadoop.hdfs.server.namenode.FSDirStatAndListingOp.getBlockLocations(FSDirStatAndListingOp.java:159)
	at org.apache.hadoop.hdfs.server.namenode.FSNamesystem.getBlockLocations(FSNamesystem.java:2040)
	at org.apache.hadoop.hdfs.server.namenode.NameNodeRpcServer.getBlockLocations(NameNodeRpcServer.java:738)
	at org.apache.hadoop.hdfs.protocolPB.ClientNamenodeProtocolServerSideTranslatorPB.getBlockLocations(ClientNamenodeProtocolServerSideTranslatorPB.java:454)
	at org

In [9]:
ip_pdf.plot.barh(x="client_ip", y="requests", legend=False, color="coral", figsize=(7, 4))
plt.title("Top 10 client IPs")
plt.xlabel("requests")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

NameError: name 'ip_pdf' is not defined

In [10]:
method_pdf = logs.groupBy("method").count().orderBy(F.desc("count")).toPandas()

method_pdf.plot.bar(x="method", y="count", legend=False, color="seagreen", figsize=(6, 3))
plt.title("HTTP method")
plt.tight_layout()
plt.show()

26/10/04 02:33:29 882 ERROR TaskSetManager: Task 1 in stage 6.0 failed 4 times; aborting job


Py4JJavaError: An error occurred while calling o136.collectToPython.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 1 in stage 6.0 failed 4 times, most recent failure: Lost task 1.3 in stage 6.0 (TID 91) (172.42.2.173 executor 52): java.io.FileNotFoundException: 
File does not exist: /data/logs/httpd/access.log_2026-10-03-20-30
	at org.apache.hadoop.hdfs.server.namenode.INodeFile.valueOf(INodeFile.java:87)
	at org.apache.hadoop.hdfs.server.namenode.INodeFile.valueOf(INodeFile.java:77)
	at org.apache.hadoop.hdfs.server.namenode.FSDirStatAndListingOp.getBlockLocations(FSDirStatAndListingOp.java:159)
	at org.apache.hadoop.hdfs.server.namenode.FSNamesystem.getBlockLocations(FSNamesystem.java:2040)
	at org.apache.hadoop.hdfs.server.namenode.NameNodeRpcServer.getBlockLocations(NameNodeRpcServer.java:738)
	at org.apache.hadoop.hdfs.protocolPB.ClientNamenodeProtocolServerSideTranslatorPB.getBlockLocations(ClientNamenodeProtocolServerSideTranslatorPB.java:454)
	at org

---

## （オプション）デモ3の Hive テーブルを同じ Notebook から触る

`demo_logs.weblog` を RegexSerDe で作成済みの場合、**SQL 1 行** でも同じデータを参照できます。PySpark と Hive の両方を 1 画面で見せると「分析のやり方の選択肢が多い」ことを説明しやすいです。

In [ ]:
# テーブル未作成の場合はスキップしてください
hive_sample = spark.sql("""
    SELECT client_ip, method, path, status, log_datetime
    FROM demo_logs.weblog
    LIMIT 10
""")

hive_sample.toPandas()

---

### 追加でアピールしやすい Notebook アイデア（必要に応じて別 Notebook 化）

| アイデア | 内容 | 向いている理由 |
|---------|------|----------------|
| **対話的しきい値** | `STATUS_MIN = 400` など変数を変えて再実行 | 「コードは短いがパラメータで調査を回せる」 |
| **特定 IP の追跡** | `TARGET_IP = "..."` で filter のみ | インシデント調査に近いイメージ |
| **%sql マagic** | Hive カーネルまたは spark.sql をセル分割 | SQL 派の分析者向け |

本 Notebook は **デモ2の py ファイルと同じロジック** をセル分割した構成です。ターミナルデモの続きとして「同じ処理を UI で試せる」と説明できます。